# 03. RetinaNet (one-stage, anchor-based)

Секція 1 — аналіз якорів на розмітці train: без моделі й без GPU, лише рамки з `train_coco.json`. Результат — вибір розмірів і форм якорів для експерименту 1. Навчання додамо наступними секціями.

## 1. Об'єкти очима моделі

torchvision стискає кадр так, щоб довша сторона стала 960 (`min(960 / коротка, 960 / довга)`), і доповнює до кратного 32. Тому всі рамки переводимо в цей масштаб: саме такими їх бачать якорі.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

IMG_SIZE = 960
SEED = 42
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    PREPARED = next(Path("/kaggle/input").glob("**/hw2_prepared/train_coco.json")).parent
else:
    PREPARED = Path.cwd().parent.parent / "hw2_prepared"

coco = json.load(open(PREPARED / "train_coco.json"))
images = pd.DataFrame(coco["images"]).set_index("id")
classes = {c["id"]: c["name"] for c in coco["categories"]}
scale = np.minimum(IMG_SIZE / images[["width", "height"]].min(axis=1), IMG_SIZE / images[["width", "height"]].max(axis=1))

boxes = pd.DataFrame(coco["annotations"])
s = boxes["image_id"].map(scale).to_numpy()
boxes[["x", "y", "w", "h"]] = np.stack(boxes["bbox"]) * s[:, None]
for side, col in (("W", "width"), ("H", "height")):
    boxes[side] = (np.ceil(np.floor(boxes["image_id"].map(images[col]).to_numpy() * s) / 32) * 32).astype(int)
boxes["cls"] = boxes["category_id"].map(classes)
boxes["size"] = np.sqrt(boxes["w"] * boxes["h"])
boxes["ratio"] = boxes["h"] / boxes["w"]
SIZE_BINS = [0, 8, 16, 32, 64, np.inf]
boxes["size_bin"] = pd.cut(boxes["size"], SIZE_BINS, right=False)
print(len(images), "кадрів,", len(boxes), "об'єктів")

4868 кадрів, 222879 об'єктів


In [2]:
q = [0.1, 0.5, 0.9]
size_ratio = pd.concat([boxes.groupby("cls")[["size", "ratio"]].quantile(q).unstack(),
                        boxes[["size", "ratio"]].quantile(q).unstack().to_frame("усі").T])
size_ratio.insert(0, ("n", ""), boxes["cls"].value_counts().reindex(size_ratio.index).fillna(len(boxes)).astype(int))
display(size_ratio.round(2))
display((pd.crosstab(boxes["cls"], boxes["size_bin"], normalize="index", margins=True) * 100).round(1))

n   size                ratio            
                    0.1    0.5     0.9   0.1   0.5   0.9
bicycle     8129  11.41  18.94   33.03  0.57  1.05  1.95
bus         1945  20.78  52.09  118.25  0.34  0.97  1.60
car       101814  13.86  26.93   57.50  0.41  0.80  1.22
motor      20712  11.02  19.21   33.00  0.64  1.24  1.96
person     66353  10.12  18.42   30.97  1.32  2.00  2.73
tricycle    8337  14.31  28.79   53.40  0.54  0.94  1.48
truck       6137  18.13  37.04   84.89  0.45  0.93  1.29
van         9452  15.50  25.43   58.14  0.51  0.90  1.40
усі       222879  12.06  22.77   49.73  0.47  1.06  2.26

size_bin,"[0.0, 8.0)","[8.0, 16.0)","[16.0, 32.0)","[32.0, 64.0)","[64.0, inf)"
cls,,,,,
bicycle,1.8,34.6,52.5,11.1,0.0
bus,0.0,5.0,23.1,32.3,39.5
car,0.5,15.8,46.4,30.6,6.7
motor,1.9,30.5,56.2,10.8,0.6
person,4.2,32.8,54.1,8.4,0.5
tricycle,0.0,15.6,44.6,34.5,5.4
truck,0.0,5.2,35.1,40.2,19.4
van,0.0,10.9,52.6,28.9,7.6
All,1.7,22.3,49.5,21.8,4.7


## 2. Покриття об'єктів якорями

Повторюємо, як torchvision `AnchorGenerator` ставить якорі: у кожній точці рівня P3–P7 (крок = розмір картинки // розмір карти) — `розміри × форми` якорів, форма = висота / ширина, кути округлені. Для кожного об'єкта рахуємо:

- `best_iou` — найбільший IoU з будь-яким якорем;
- `npos` — скільки якорів мають IoU ≥ 0,5 (стануть «об'єктами» під час навчання);
- `covered` — чи є хоча б один такий якір. Інакше об'єкт вчиться лише через правило «найкращий якір».

Щоб не тримати всі 98 тис. якорів на кадр, для кожного об'єкта беремо лише точки сітки біля його центру: якір з IoU ≥ 0,5 не може стояти далі, ніж на свою ширину (висоту) від центру об'єкта.

Спрощення: кожен об'єкт рахуємо окремо, тобто не враховуємо, що в густій сцені якір може дістатися сусідньому об'єкту з більшим IoU.

In [3]:
def retinanet_sizes(factor=1.0):
    return [tuple(int(x * factor * 2 ** (k / 3)) for k in range(3)) for x in (32, 64, 128, 256, 512)]


def anchor_match(b, sizes, ratios, chunk=20000):
    x1, y1, w, h = (b[c].to_numpy(np.float32) for c in ("x", "y", "w", "h"))
    W, H = b["W"].to_numpy(), b["H"].to_numpy()
    best = np.zeros(len(b), np.float32)
    npos = np.zeros(len(b), np.int32)
    r = np.sqrt(np.asarray(ratios, np.float32))
    for level, level_sizes in enumerate(sizes):
        gw, gh = np.ceil(W / 2 ** (level + 3)), np.ceil(H / 2 ** (level + 3))
        sx, sy = (W // gw).astype(np.float32), (H // gh).astype(np.float32)
        ws = (np.asarray(level_sizes, np.float32)[None] / r[:, None]).ravel()
        hs = (np.asarray(level_sizes, np.float32)[None] * r[:, None]).ravel()
        for hw, hh in zip(np.round(ws / 2), np.round(hs / 2)):
            ox = np.arange(-int(np.ceil(2 * hw / sx.min())) - 1, int(np.ceil(2 * hw / sx.min())) + 2)
            oy = np.arange(-int(np.ceil(2 * hh / sy.min())) - 1, int(np.ceil(2 * hh / sy.min())) + 2)
            for i in range(0, len(b), chunk):
                j = slice(i, i + chunk)
                ix = np.round((x1[j] + w[j] / 2) / sx[j])[:, None] + ox
                iy = np.round((y1[j] + h[j] / 2) / sy[j])[:, None] + oy
                ax, ay = ix * sx[j, None], iy * sy[j, None]
                iw = np.clip(np.minimum(ax + hw, (x1[j] + w[j])[:, None]) - np.maximum(ax - hw, x1[j, None]), 0, None)
                ih = np.clip(np.minimum(ay + hh, (y1[j] + h[j])[:, None]) - np.maximum(ay - hh, y1[j, None]), 0, None)
                iw *= (ix >= 0) & (ix < gw[j, None])
                ih *= (iy >= 0) & (iy < gh[j, None])
                inter = iw[:, :, None] * ih[:, None, :]
                iou = inter / ((w[j] * h[j])[:, None, None] + 4 * hw * hh - inter)
                best[j] = np.maximum(best[j], iou.max(axis=(1, 2)))
                npos[j] += (iou >= 0.5).sum(axis=(1, 2))
    return pd.DataFrame({"best_iou": best, "npos": npos, "covered": npos > 0}, index=b.index)


def coverage_table(results):
    rows = {}
    for name, m in results.items():
        row = {"covered_%": m["covered"].mean() * 100, "best_iou": m["best_iou"].mean(), "npos": m["npos"].mean()}
        row |= (m["covered"].groupby(boxes["size_bin"], observed=False).mean() * 100).rename(lambda b: f"cov_{b}").to_dict()
        rows[name] = row
    return pd.DataFrame(rows).T.round(2)

In [ ]:
DATA_RATIOS = tuple(round(float(r), 2) for r in boxes["ratio"].quantile([1 / 6, 1 / 2, 5 / 6]))
FACTORS = (1.0, 0.75, 0.5, 0.375, 0.25)
CONFIGS = {f"x{f} {name}": (retinanet_sizes(f), ratios)
           for f in FACTORS for name, ratios in (("r0.5-1-2", (0.5, 1.0, 2.0)), (f"r{'-'.join(map(str, DATA_RATIOS))}", DATA_RATIOS))}
results = {name: anchor_match(boxes, *cfg) for name, cfg in CONFIGS.items()}
print("форми з даних (1/6, 1/2, 5/6 квантилі висота/ширина):", DATA_RATIOS)
display(pd.Series({f"x{f}": " | ".join("/".join(map(str, level)) for level in retinanet_sizes(f)) for f in FACTORS}, name="розміри P3 | … | P7").to_frame())
coverage = coverage_table(results)
coverage

In [ ]:
by_class = pd.DataFrame({name: m["covered"].groupby(boxes["cls"]).mean() * 100 for name, m in results.items()}).T
by_class.round(1)

## 3. k-means по формах рамок (YOLOv2)

Метод YOLOv2: k-means по (ширина, висота) рамок з відстанню `1 − IoU` (рамки суміщені центрами), щоб великі рамки не важили більше за малі. Якість набору шаблонів — середній IoU кожного об'єкта з найближчим шаблоном (`avg_iou`, як у Table 1 статті). Тут k = 9 — стільки ж шаблонів, скільки якорів в одній точці RetinaNet. Порівнюємо з якорями з секції 2, теж без урахування положення на сітці.

In [ ]:
def shape_iou(wh, priors):
    inter = np.minimum(wh[:, None, 0], priors[None, :, 0]) * np.minimum(wh[:, None, 1], priors[None, :, 1])
    return inter / (wh.prod(1)[:, None] + priors.prod(1)[None] - inter)


def kmeans_iou(wh, k, iters=100, seed=SEED):
    centers = wh[np.random.default_rng(seed).choice(len(wh), k, replace=False)]
    for _ in range(iters):
        nearest = shape_iou(wh, centers).argmax(1)
        updated = np.array([np.median(wh[nearest == c], axis=0) if (nearest == c).any() else centers[c] for c in range(k)])
        if np.allclose(updated, centers):
            break
        centers = updated
    return centers[np.argsort(centers.prod(1))]


def anchor_shapes(sizes, ratios):
    r = np.sqrt(np.asarray(ratios, np.float32))
    return np.array([(2 * np.round(s / q / 2), 2 * np.round(s * q / 2)) for level in sizes for s in level for q in r])


wh = boxes[["w", "h"]].to_numpy(np.float32)
centers = kmeans_iou(wh, 9)
nearest = shape_iou(wh, centers).argmax(1)
display(pd.DataFrame({"w": centers[:, 0], "h": centers[:, 1], "size": np.sqrt(centers.prod(1)), "ratio": centers[:, 1] / centers[:, 0],
                      "share_%": np.bincount(nearest, minlength=9) / len(wh) * 100}).round(1))
avg_iou = {"k-means 9": shape_iou(wh, centers).max(1).mean()}
avg_iou |= {name: shape_iou(wh, anchor_shapes(*cfg)).max(1).mean() for name, cfg in CONFIGS.items()}
pd.Series(avg_iou, name="avg_iou").round(3).to_frame()

## Джерела

**Бібліотеки**
- NumPy, pandas — обчислення й таблиці.

**Відтворено з torchvision (без копіювання коду)**
- Розміри якорів RetinaNet `(x, x·2^(1/3), x·2^(2/3))` для x = 32…512 на P3–P7 і форми (0,5; 1; 2) — `torchvision/models/detection/retinanet.py::_default_anchorgen`.
- Побудова якорів: форма = висота / ширина, `w = розмір / √форма`, `h = розмір · √форма`, округлення кутів, центри в `arange(розмір карти) · крок`, крок = розмір картинки // розмір карти — `torchvision/models/detection/anchor_utils.py::AnchorGenerator`: https://github.com/pytorch/vision/blob/main/torchvision/models/detection/anchor_utils.py
- Масштаб `min(min_size / коротка, max_size / довга)` і доповнення до кратного 32 — `torchvision/models/detection/transform.py::GeneralizedRCNNTransform`, там само.
- Пороги IoU 0,5 / 0,4 — `RetinaNet(fg_iou_thresh=0.5, bg_iou_thresh=0.4)`, `retinanet.py`.

**Методи зі статей**
- k-means з відстанню `1 − IoU(рамка, центр)` і метрика «середній IoU з найближчим шаблоном» — J. Redmon, A. Farhadi, *YOLO9000: Better, Faster, Stronger*, 2017, розділ «Dimension Clusters», Table 1: https://arxiv.org/abs/1612.08242
- RetinaNet, якорі на P3–P7 — T.-Y. Lin et al., *Focal Loss for Dense Object Detection*, 2017: https://arxiv.org/abs/1708.02002

**Наші рішення**
- Метрика покриття (`covered`, `npos`, `best_iou`) з урахуванням положення якорів на сітці; пошук лише в околі центру об'єкта.
- Набори для порівняння: стандартні розміри × множник 1…0,25; форми зі стандартних або з квантилів 1/6, 1/2, 5/6 нашого відношення висота / ширина; k = 9 у k-means; центр кластера — медіана.